# Opik Metrics Explorer

Notebook para carregar traces e threads do Opik, escolher colunas/metricas e calcular agregacoes.

Baseado na API oficial do SDK:
- `client.search_traces(project_name=..., filter_string=..., max_results=..., truncate=..., exclude=...)`
- `client.search_threads(project_name=..., filter_string=..., max_results=..., truncate=...)`
- `client.log_threads_feedback_scores(scores=[...], project_name=...)` para feedback por thread

Os filtros suportam campos como `thread_id`, `tags`, `metadata.*`, `feedback_scores.*`, `usage.*`, `duration` e outros.

In [ ]:
from __future__ import annotations

import os
from typing import Any, Iterable

import pandas as pd

try:
    from opik import Opik
except Exception as exc:
    raise RuntimeError("The opik package is required in this environment.") from exc

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 200)

In [ ]:
PROJECT_NAME = os.getenv('OPIK_PROJECT_NAME', 'ChatDev')
OPIK_HOST = os.getenv('OPIK_URL_OVERRIDE', 'https://www.comet.com/opik/api')
OPIK_WORKSPACE = os.getenv('OPIK_WORKSPACE') or os.getenv('OPIK_WORKSPACE_NAME')
OPIK_API_KEY = os.getenv('OPIK_API_KEY')

client = Opik(
    project_name=PROJECT_NAME,
    host=OPIK_HOST,
    workspace=OPIK_WORKSPACE,
    api_key=OPIK_API_KEY,
)

PROJECT_NAME

In [ ]:
def flatten_dict(value: dict[str, Any], prefix: str = '', sep: str = '.') -> dict[str, Any]:
    items: dict[str, Any] = {}
    for key, item in value.items():
        new_key = f'{prefix}{sep}{key}' if prefix else key
        if isinstance(item, dict):
            items.update(flatten_dict(item, new_key, sep=sep))
        else:
            items[new_key] = item
    return items

def pick_fields(record: Any, fields: Iterable[str]) -> dict[str, Any]:
    data: dict[str, Any] = {}
    for field in fields:
        value = record
        for part in field.split('.'):
            if value is None:
                break
            if isinstance(value, dict):
                value = value.get(part)
            else:
                value = getattr(value, part, None)
        data[field] = value
    return data

def as_dict(record: Any) -> dict[str, Any]:
    if isinstance(record, dict):
        return record
    if hasattr(record, 'model_dump'):
        return record.model_dump()
    if hasattr(record, 'dict'):
        return record.dict()
    return {k: getattr(record, k) for k in dir(record) if not k.startswith('_')}

In [ ]:
def fetch_traces(
    filter_string: str | None = None,
    max_results: int = 1000,
    truncate: bool = False,
    exclude: list[str] | None = None,
) -> pd.DataFrame:
    traces = client.search_traces(
        project_name=PROJECT_NAME,
        filter_string=filter_string,
        max_results=max_results,
        truncate=truncate,
        exclude=exclude,
    )
    rows = [as_dict(trace) for trace in traces]
    return pd.json_normalize(rows, sep='.') if rows else pd.DataFrame()

def fetch_threads(
    filter_string: str | None = None,
    max_results: int = 1000,
    truncate: bool = False,
) -> pd.DataFrame:
    threads = client.search_threads(
        project_name=PROJECT_NAME,
        filter_string=filter_string,
        max_results=max_results,
        truncate=truncate,
    )
    rows = [as_dict(thread) for thread in threads]
    return pd.json_normalize(rows, sep='.') if rows else pd.DataFrame()

In [ ]:
trace_df = fetch_traces(max_results=200, truncate=False)
thread_df = fetch_threads(max_results=200, truncate=False)

print('Trace columns:', list(trace_df.columns))
print('Thread columns:', list(thread_df.columns))

if not trace_df.empty:
    print('\nSample trace row (first):')
    print(trace_df.iloc[0].to_dict())

def nested_keys(df: pd.DataFrame, col: str) -> list[str]:
    keys = set()
    if col in df.columns:
        for value in df[col].dropna():
            if isinstance(value, dict):
                keys.update(value.keys())
    return sorted(keys)

print('\nTrace feedback_score keys:', nested_keys(trace_df, 'feedback_scores'))
print('Trace metadata keys:', nested_keys(trace_df, 'metadata'))
print('\nThread feedback_score keys:', nested_keys(thread_df, 'feedback_scores'))

In [ ]:
TRACE_COLUMNS = [
    'id',
    'name',
    'duration',
    'usage.total_tokens',
    'usage.prompt_tokens',
    'usage.completion_tokens',
    'total_estimated_cost',
    'tags',
    'span_count',
    'llm_span_count',
    'thread_id',
    'feedback_scores',
    'metadata.node_id',
    'metadata.agent_role',
    'metadata.provider',
    'metadata.model',
]

THREAD_COLUMNS = [
    'id',
    'status',
    'duration',
    'number_of_messages',
    'created_at',
    'last_updated_at',
    'feedback_scores',
    'tags',
]

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    keep = [column for column in columns if column in df.columns]
    return df.loc[:, keep].copy() if keep else df.copy()

def feedback_scores_to_frame(df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for _, trace in df.iterrows():
        scores = trace.get('feedback_scores') or []
        if not isinstance(scores, list):
            continue
        for score in scores:
            if not isinstance(score, dict):
                continue
            rows.append({
                'trace_id': trace.get('id'),
                'thread_id': trace.get('thread_id'),
                'trace_name': trace.get('name'),
                'score_name': score.get('name'),
                'score_value': score.get('value'),
                'score_reason': score.get('reason'),
                'score_source': score.get('source'),
                'score_created_at': score.get('created_at'),
                'agent_role': trace.get('metadata.agent_role'),
                'node_id': trace.get('metadata.node_id'),
                'provider': trace.get('metadata.provider'),
                'model': trace.get('metadata.model'),
                'tags': trace.get('tags'),
            })
    scores_df = pd.DataFrame(rows)
    if not scores_df.empty:
        scores_df['score_value'] = pd.to_numeric(scores_df['score_value'], errors='coerce')
    return scores_df

trace_df = select_columns(fetch_traces(max_results=200, truncate=False), TRACE_COLUMNS)
thread_df = select_columns(fetch_threads(max_results=200, truncate=False), THREAD_COLUMNS)
feedback_df = feedback_scores_to_frame(trace_df)

trace_df.head(10), thread_df.head(10), feedback_df.head(10)

In [ ]:
# Inspect and aggregate feedback scores
print('Feedback score columns:', list(feedback_df.columns))
print('Score names:', sorted(feedback_df['score_name'].dropna().unique().tolist()) if not feedback_df.empty else [])

if not feedback_df.empty:
    feedback_summary = feedback_df.groupby('score_name', dropna=False)['score_value'].agg(['count', 'mean', 'min', 'max']).sort_values('mean', ascending=False)
    score_by_role = feedback_df.groupby(['agent_role', 'score_name'], dropna=False)['score_value'].agg(['count', 'mean']).reset_index()
    wide_feedback = feedback_df.pivot_table(index=['trace_id', 'thread_id'], columns='score_name', values='score_value', aggfunc='first').reset_index()
else:
    feedback_summary = pd.DataFrame()
    score_by_role = pd.DataFrame()
    wide_feedback = pd.DataFrame()

feedback_summary, score_by_role.head(20), wide_feedback.head(20)

## How to select fields

Use `fetch_traces(filter_string=...)` or `fetch_threads(filter_string=...)` with OQL filters.
Examples:
- `thread_id = "thread_123"`
- `metadata.agent_role = "Client Liaison"`
- `feedback_scores.Responsiveness > 0.8`
- `tags contains "production"`

To choose columns, edit `TRACE_COLUMNS` and `THREAD_COLUMNS`, then rerun the selection cells.